In [1]:
import pandas as pd
import ast
from gensim import corpora
df = pd.read_csv("data_bersih.csv")

print(df.columns.tolist())
print("Jumlah baris:", len(df))

['url', 'judul', 'konten', 'sumber', 'tanggal', 'clean', 'Token', 'Normalisasi', 'stopwords', 'stemmer', 'Berita']
Jumlah baris: 194


In [2]:
df['tokens'] = df['Token'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Cek contoh hasil
print(df['tokens'].head(3))
print("Jumlah dokumen:", len(df))

# --- 3️⃣ Buat dictionary dan corpus untuk LDA ---
dictionary = corpora.Dictionary(df['tokens'])

# Filter kata (optional, bisa sesuaikan)
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in df['tokens']]

print("Jumlah kata unik (vocabulary):", len(dictionary))
print("Contoh corpus dokumen pertama:", corpus[0][:10])


0    [karena,  , distribusi,  , logistik,  , pemilu...
1    [sumenep,  , radarmaduraid,  , kejaksaan,  , n...
2    [kbrn,  , maros,  , komisi,  , pemilihan,  , u...
Name: tokens, dtype: object
Jumlah dokumen: 194
Jumlah kata unik (vocabulary): 112
Contoh corpus dokumen pertama: [(0, 2), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 2), (8, 1), (9, 1)]


In [3]:
from gensim.models import LdaModel

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    passes=10,
    random_state=42
)

for idx, topic in lda_model.print_topics(-1):
    print(f"Topik {idx+1}: {topic}\n")


Topik 1: 0.136*"logistik" + 0.080*"pemilu" + 0.061*"distribusi" + 0.030*"untuk" + 0.029*"dan" + 0.029*"pilkada" + 0.027*"memastikan" + 0.027*"pendistribusian" + 0.024*"tahun" + 0.022*"kapolres"

Topik 2: 0.084*"logistik" + 0.077*"pemilu" + 0.051*"kpu" + 0.044*"bawaslu" + 0.044*"pemilihan" + 0.038*"umum" + 0.033*"untuk" + 0.029*"distribusi" + 0.027*"pendistribusian" + 0.027*"komisi"

Topik 3: 0.086*"pemilihan" + 0.083*"kpu" + 0.067*"umum" + 0.065*"komisi" + 0.055*"suara" + 0.050*"surat" + 0.036*"kabupaten" + 0.034*"kota" + 0.028*"pemilu" + 0.024*"dalam"

Topik 4: 0.091*"di" + 0.091*"pemilu" + 0.072*"logistik" + 0.064*"kpu" + 0.043*"umum" + 0.041*"pemilihan" + 0.038*"kota" + 0.032*"suara" + 0.032*"ke" + 0.031*"gudang"

Topik 5: 0.088*"suara" + 0.078*"dan" + 0.072*"surat" + 0.051*"yang" + 0.049*"kpu" + 0.042*"pemilu" + 0.035*"logistik" + 0.027*"untuk" + 0.027*"di" + 0.024*"rusak"



# Coherence & pemilihan jumlah topik terbaik

In [4]:
from gensim.models import CoherenceModel, LdaModel

def train_lda(k, corpus, dictionary, texts):
    model = LdaModel(
        corpus=corpus, id2word=dictionary, num_topics=k,
        random_state=42, passes=10, iterations=200,
        alpha='auto', eta='auto', chunksize=2000, eval_every=None
    )
    coh = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v').get_coherence()
    return model, coh

k_list = [4,5,6,7,8,10]
results = []
best = (None, -1, None)  # (k, coherence, model)

for k in k_list:
    m, c = train_lda(k, corpus, dictionary, df['tokens'])
    results.append((k, c))
    if c > best[1]:
        best = (k, c, m)

print("Coherence per K:", results)
print(f"Best K = {best[0]} | Coherence = {best[1]:.3f}")

lda_model = best[2]   # pakai model terbaik ke depannya


Coherence per K: [(4, np.float64(0.2950846671157693)), (5, np.float64(0.2891202493686511)), (6, np.float64(0.3227260049724365)), (7, np.float64(0.30146800996598794)), (8, np.float64(0.30994227086079473)), (10, np.float64(0.33443845397317157))]
Best K = 10 | Coherence = 0.334


# Visualisasi interaktif topik (pyLDAvis)

In [5]:
# !pip install pyLDAvis  # jalankan jika belum terpasang
import pyLDAvis, pyLDAvis.gensim_models as gensimvis


# 3. Aktifkan mode notebook agar tampil langsung di bawah cell
pyLDAvis.enable_notebook()

# 4. Siapkan visualisasi interaktif
vis = gensimvis.prepare(
    lda_model,      # model LDA kamu
    corpus,         # corpus (bag of words)
    dictionary,     # kamus kata
    sort_topics=False
)

# 5. Tampilkan langsung di notebook
vis



PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.029983 -0.028908       1        1   6.886614
1      0.101528  0.037433       2        1  11.865201
2      0.091439 -0.102950       3        1  21.407136
3      0.003114  0.048957       4        1   8.708699
4      0.067013 -0.141630       5        1  18.978948
5     -0.101431  0.013897       6        1   3.133950
6      0.025623 -0.009654       7        1   3.165854
7     -0.096836  0.181761       8        1   6.825606
8      0.149173  0.084464       9        1  17.123260
9     -0.269607 -0.083368      10        1   1.904731, topic_info=        Term       Freq      Total Category  logprob  loglift
4   logistik  98.000000  98.000000  Default  30.0000  30.0000
0        dan  54.000000  54.000000  Default  29.0000  29.0000
42        di  46.000000  46.000000  Default  28.0000  28.0000
36     surat  64.000000  64.000000  Default  27.0000  27.0000
7      suara  84.000000  84.000000  Default  26.0000  26.0000
..       ...        ...        ...      ...      ...      ...
7      suara   0.085957  84.538188  Topic10  -5.9769  -2.9303
99     timur   0.085953   6.537045  Topic10  -5.9769  -0.3706
6     pemilu   0.085953  98.384627  Topic10  -5.9769  -3.0820
4   logistik   0.085953  98.118242  Topic10  -5.9769  -3.0793
90   bandung   0.085948   7.575366  Topic10  -5.9770  -0.5181

[413 rows x 6 columns], token_table=      Topic      Freq  Term
term                       
81        3  0.603837   ada
81        5  0.120767   ada
81        6  0.120767   ada
67        1  0.264803  akan
67        5  0.529606  akan
...     ...       ...   ...
9         6  0.084053  yang
9         7  0.028018  yang
9         8  0.084053  yang
9         9  0.028018  yang
9        10  0.028018  yang

[406 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Dominant topic per dokumen + label topik + ekspor

In [8]:
def dominant_topic(bow, model):
    topics = model.get_document_topics(bow, minimum_probability=0)
    topics = sorted(topics, key=lambda x: x[1], reverse=True)
    return topics[0]  # (topic_id, prob)

dom = [dominant_topic(b, lda_model) for b in corpus]
df['dominant_topic'] = [t[0] + 1 for t in dom]  # mulai dari 1
df['topic_prob'] = [t[1] for t in dom]

# Buat label topik dari 6 kata teratas
topic_labels = {}
for i in range(lda_model.num_topics):
    words = [w for w,_ in lda_model.show_topic(i, topn=6)]
    topic_labels[i+1] = ", ".join(words)

df['topic_label'] = df['dominant_topic'].map(topic_labels)

# Simpan untuk analisis/visualisasi lanjutan
cols_export = [c for c in ['id','judul','sumber','tanggal','tweet'] if c in df.columns] + \
              ['dominant_topic','topic_prob','topic_label']
df[cols_export].to_csv("hasil_topik_per_dokumen.csv", index=False)
print("Saved: hasil_topik_per_dokumen.csv")


Saved: hasil_topik_per_dokumen.csv
